In [1]:
import os
import sys
from importlib.machinery import SourceFileLoader
import torch
import json 
import torch.nn.functional as F
def find_root_path(path:str, word:str):
    parts = path.split(word, 1)    
    return parts[0] + word if len(parts) > 1 else path 

try:
    current_path = os.path.abspath(__file__)
except NameError:
    current_path = os.getcwd()
    
root_folder = find_root_path(os.path.abspath(current_path), 'art_lang')
sys.path.append(root_folder)

# from src 
from rpod.decision_transformer.adapter import FrozenTextAdapter  
device =  "cuda" if torch.cuda.is_available() else "cpu"


%load_ext autoreload
%autoreload 2

### Cosine similarity matrix (RPOD) 

In [17]:
def compute_cross_group_similarity(text_encoder, command_data, device):
    """
    Compute 6x6 cosine similarity matrix between command mode groups.
    Each group has 'descriptions': list[str].
    Returns: torch.Tensor [6, 6] with mean cosine similarities.
    """
    group_embs = []
    with torch.inference_mode():
        # Encode and mean-pool each group's text
        for group in command_data[:6]:
            descs = group["description"]
            out = text_encoder(descs)                 # [B, T, H]
            emb = out.mean(dim=1)                     # mean-pool tokens → [B, H]
            emb = F.normalize(emb, dim=1)             # normalize for cosine sim
            group_embs.append(emb.mean(dim=0))        # mean over group → [H]
        group_embs = torch.stack(group_embs)          # [6, H]
    
    # 6x6 cosine similarity matrix
    sim_mat = group_embs @ group_embs.T               # [6, 6]
    return sim_mat.cpu()

In [19]:
# Load Text encoder (For the rendezvous)
MODEL = os.getenv("FTA_MODEL", "distilbert-base-uncased")   # this is encoder only 
text_encoder = FrozenTextAdapter(model_name=MODEL, out_dim=384, output_mode="tokens").to(device).eval()

# Load command summary data
command_data = []
with open(root_folder+"/rpod/dataset/commands_summary.jsonl", "r") as f:
    for line in f:
        command_data.append(json.loads(line))
        
sim_matrix = compute_cross_group_similarity(text_encoder, command_data, device)
print(sim_matrix)

tensor([[0.8478, 0.8177, 0.8037, 0.8012, 0.7783, 0.8153],
        [0.8177, 0.8966, 0.8092, 0.8155, 0.7699, 0.8261],
        [0.8037, 0.8092, 0.9068, 0.8533, 0.7732, 0.8024],
        [0.8012, 0.8155, 0.8533, 0.8756, 0.7888, 0.8115],
        [0.7783, 0.7699, 0.7732, 0.7888, 0.8927, 0.7993],
        [0.8153, 0.8261, 0.8024, 0.8115, 0.7993, 0.8570]])


### Cosine Similarity Matrix (Freeflyer) 

In [21]:
def compute_cross_group_similarity2(text_encoder, command_data, device):
    """
    Compute 6x6 cosine similarity matrix between command mode groups.
    Each group has 'descriptions': list[str].
    Returns: torch.Tensor [6, 6] with mean cosine similarities.
    """
    group_embs = []
    with torch.inference_mode():
        # Encode and mean-pool each group's text
        for descs in command_data[:6]:
            out = text_encoder(descs)                 # [B, T, H]
            emb = out.mean(dim=1)                     # mean-pool tokens → [B, H]
            emb = F.normalize(emb, dim=1)             # normalize for cosine sim
            group_embs.append(emb.mean(dim=0))        # mean over group → [H]
        group_embs = torch.stack(group_embs)          # [6, H]
    
    # 6x6 cosine similarity matrix
    sim_mat = group_embs @ group_embs.T               # [6, 6]
    return sim_mat.cpu()

In [22]:
with open(root_folder+"/rpod/dataset/master_file_freeflyer.jsonl", "r") as f:
    data = json.load(f)  # not line-by-line!
    
    command_data = []
    for k in sorted(data.keys(), key=lambda s: int(s)):
        texts = [entry["text"] for entry in data[k] if "text" in entry]
        command_data.append(texts)

sim_matrix = compute_cross_group_similarity2(text_encoder, command_data, device)
print(sim_matrix)

tensor([[0.8911, 0.8901, 0.8893, 0.8922, 0.8717, 0.8746],
        [0.8901, 0.9009, 0.8900, 0.9022, 0.8619, 0.8690],
        [0.8893, 0.8900, 0.8925, 0.8952, 0.8712, 0.8743],
        [0.8922, 0.9022, 0.8952, 0.9085, 0.8661, 0.8733],
        [0.8717, 0.8619, 0.8712, 0.8661, 0.9139, 0.9078],
        [0.8746, 0.8690, 0.8743, 0.8733, 0.9078, 0.9065]])
